In [ ]:
!pip install psycopg2-binary scikit-learn pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.8 MB/s eta 0:00:00


In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="aws-1-ap-southeast-1.pooler.supabase.com",
    port="6543",
    dbname="postgres",
    user="postgres.guaovznwrmjjsgewhhip",
    password="Riyazshaik@2501"
)

df = pd.read_sql("SELECT * FROM mart_transactions", conn)
print(df.shape)
df.head()

/tmp/ipykernel_7186/584246725.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM mart_transactions", conn)


(5100, 12)


,transaction_id,transaction_date,department,vendor,payment_type,amount,transaction_month,transaction_year,is_round_number_flag,is_spend_spike_flag,vendor_amount_occurrence,is_potential_duplicate_flag
0,TXN12056,2024-09-20,Engineering,Aguilar-Harris,Credit Card,160.14,9.0,2024.0,False,False,1,False
1,TXN12904,2024-03-06,Engineering,Aguilar-Harris,Check,162.52,3.0,2024.0,False,False,1,False
2,TXN11219,2024-07-14,Operations,Aguilar-Harris,Check,168.87,7.0,2024.0,False,False,1,False
3,TXN12565,2024-03-28,Sales,Aguilar-Harris,Wire Transfer,218.86,3.0,2024.0,False,False,1,False
4,TXN10368,2024-12-09,Marketing,Aguilar-Harris,Check,247.21,12.0,2024.0,False,False,1,False


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Prepare features for the model
df_model = df.copy()

# Encode categorical columns
le_dept = LabelEncoder()
le_vendor = LabelEncoder()
le_payment = LabelEncoder()

df_model['department_encoded'] = le_dept.fit_transform(df_model['department'])
df_model['vendor_encoded'] = le_vendor.fit_transform(df_model['vendor'])
df_model['payment_type_encoded'] = le_payment.fit_transform(df_model['payment_type'])

# Select features for anomaly detection
features = ['amount', 'department_encoded', 'vendor_encoded', 'payment_type_encoded', 'transaction_month']
X = df_model[features]

# Train Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # assume ~5% of transactions are anomalies
    random_state=42
)

df_model['anomaly_prediction'] = iso_forest.fit_predict(X)
df_model['anomaly_score'] = iso_forest.score_samples(X)

# Convert prediction: -1 = anomaly, 1 = normal -> make it readable
df_model['is_ml_anomaly'] = df_model['anomaly_prediction'].apply(lambda x: True if x == -1 else False)

print("Total anomalies detected:", df_model['is_ml_anomaly'].sum())
print("\nSample anomalies:")
df_model[df_model['is_ml_anomaly'] == True][['transaction_id', 'vendor', 'amount', 'department', 'anomaly_score']].head(10)

Total anomalies detected: 255

Sample anomalies:


,transaction_id,vendor,amount,department,anomaly_score
3,TXN12565,Aguilar-Harris,218.86,Sales,-0.590197
8,TXN14283,Aguilar-Harris,333.50,Sales,-0.602672
10,TXN12587,Aguilar-Harris,358.21,Sales,-0.575914
13,TXN10772,Aguilar-Harris,382.74,Marketing,-0.574223
15,TXN14385,Aguilar-Harris,424.38,Sales,-0.579025
16,TXN13661,Aguilar-Harris,478.21,Engineering,-0.598796
21,TXN11417,Aguilar-Harris,519.35,Engineering,-0.597045
34,TXN12394,Aguilar-Harris,644.81,Sales,-0.577361
36,TXN14187,Aguilar-Harris,769.22,Operations,-0.574028
42,TXN11430,Aguilar-Harris,801.34,Sales,-0.577561


In [ ]:
# Check if Aguilar-Harris is just a very common vendor in general
print("Total transactions for Aguilar-Harris:", (df_model['vendor'] == 'Aguilar-Harris').sum())
print("Total transactions for all vendors (avg):", df_model['vendor'].value_counts().mean())
print("\nTop 10 most frequent vendors:")
print(df_model['vendor'].value_counts().head(10))

print("\nDoes the model's anomaly flag overlap with your injected anomalies?")
print("Round number flag + ML anomaly overlap:", ((df_model['is_round_number_flag']) & (df_model['is_ml_anomaly'])).sum())
print("Spend spike flag + ML anomaly overlap:", ((df_model['is_spend_spike_flag']) & (df_model['is_ml_anomaly'])).sum())

Total transactions for Aguilar-Harris: 112
Total transactions for all vendors (avg): 72.85714285714286

Top 10 most frequent vendors:
vendor
Jordan-Carlson                  129
Leonard, Harrell and Mcguire    116
Estrada Inc                     114
Gomez-Flores                    114
Walters LLC                     114
Tapia-Snyder                    114
Rivas-Long                      114
Monroe, Rhodes and Valentine    113
Cruz, Rowe and Young            112
Aguilar-Harris                  112
Name: count, dtype: int64

Does the model's anomaly flag overlap with your injected anomalies?
Round number flag + ML anomaly overlap: 45
Spend spike flag + ML anomaly overlap: 77


In [ ]:
df_model.to_csv('/content/final_anomaly_results.csv', index=False)
print("Saved! Total rows:", len(df_model))

Saved! Total rows: 5100
